In [1]:
import pandas as pd
import numpy as np
import joblib

val_df = joblib.load('../data/processed/val_processed_v2.pkl')
val_df = val_df.sort_values('TransactionDT').reset_index(drop=True)

print(f"Validation set shape: {val_df.shape}")
print(f"Date range: {val_df['TransactionDT'].min()} to {val_df['TransactionDT'].max()}")

# Drift injection point: halfway through the validation set
drift_point = len(val_df) // 2

before_drift = val_df.iloc[:drift_point].copy()
after_drift = val_df.iloc[drift_point:].copy()

print(f"\nBefore drift: {before_drift.shape}, fraud rate: {before_drift['isFraud'].mean():.4f}")
print(f"After drift (unmodified so far): {after_drift.shape}, fraud rate: {after_drift['isFraud'].mean():.4f}")

Validation set shape: (118108, 441)
Date range: 12192900 to 15811131

Before drift: (59054, 441), fraud rate: 0.0313
After drift (unmodified so far): (59054, 441), fraud rate: 0.0375


In [2]:
np.random.seed(42)

after_drift_modified = after_drift.copy()

current_fraud_rate = after_drift_modified['isFraud'].mean()
target_fraud_rate = current_fraud_rate * 2.5  

legit_indices = after_drift_modified[after_drift_modified['isFraud'] == 0].index
n_to_flip = int((target_fraud_rate - current_fraud_rate) * len(after_drift_modified))
flip_indices = np.random.choice(legit_indices, size=n_to_flip, replace=False)

after_drift_modified.loc[flip_indices, 'isFraud'] = 1

fraud_mask = after_drift_modified['isFraud'] == 1
after_drift_modified.loc[fraud_mask, 'TransactionAmt'] = after_drift_modified.loc[fraud_mask, 'TransactionAmt'] * 1.8

print(f"Original fraud rate (after segment): {current_fraud_rate:.4f}")
print(f"New fraud rate (after segment, post-injection): {after_drift_modified['isFraud'].mean():.4f}")
print(f"\nFraud transaction amount - before injection mean: {after_drift.loc[fraud_mask, 'TransactionAmt'].mean():.2f}")
print(f"Fraud transaction amount - after injection mean: {after_drift_modified.loc[fraud_mask, 'TransactionAmt'].mean():.2f}")

Original fraud rate (after segment): 0.0375
New fraud rate (after segment, post-injection): 0.0937

Fraud transaction amount - before injection mean: 144.53
Fraud transaction amount - after injection mean: 260.16


In [3]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score, precision_score, recall_score

model = joblib.load('../models/xgb_fraud_model_v1.pkl')
feature_cols = model.get_booster().feature_names

# Before drift performance
X_before = before_drift[feature_cols]
y_before = before_drift['isFraud']
proba_before = model.predict_proba(X_before)[:, 1]
pred_before = model.predict(X_before)

print("--- BEFORE drift segment ---")
print(f"AUC-ROC: {roc_auc_score(y_before, proba_before):.4f}")
print(f"Precision: {precision_score(y_before, pred_before):.4f}")
print(f"Recall: {recall_score(y_before, pred_before):.4f}")

# After drift (modified) performance
X_after = after_drift_modified[feature_cols]
y_after = after_drift_modified['isFraud']
proba_after = model.predict_proba(X_after)[:, 1]
pred_after = model.predict(X_after)

print("\n--- AFTER drift segment (injected) ---")
print(f"AUC-ROC: {roc_auc_score(y_after, proba_after):.4f}")
print(f"Precision: {precision_score(y_after, pred_after):.4f}")
print(f"Recall: {recall_score(y_after, pred_after):.4f}")

--- BEFORE drift segment ---
AUC-ROC: 0.9070
Precision: 0.2189
Recall: 0.7223

--- AFTER drift segment (injected) ---
AUC-ROC: 0.6879
Precision: 0.2871
Recall: 0.3592


In [5]:
import os

os.makedirs('../data/drift_simulation', exist_ok=True)

joblib.dump(before_drift, '../data/drift_simulation/before_drift.pkl', compress=3)
joblib.dump(after_drift_modified, '../data/drift_simulation/after_drift.pkl', compress=3)

# Save metadata about the injection for documentation/reproducibility
drift_metadata = {
    'drift_point_row_index': drift_point,
    'drift_point_transaction_dt': int(val_df.iloc[drift_point]['TransactionDT']),
    'fraud_rate_multiplier': 2.5,
    'amount_multiplier_for_fraud': 1.8,
    'random_seed': 42,
    'before_fraud_rate': float(before_drift['isFraud'].mean()),
    'after_fraud_rate': float(after_drift_modified['isFraud'].mean()),
    'before_auc': float(roc_auc_score(y_before, proba_before)),
    'after_auc': float(roc_auc_score(y_after, proba_after)),
}

import json
with open('../data/drift_simulation/drift_metadata.json', 'w') as f:
    json.dump(drift_metadata, f, indent=2)

print("Saved.")
print(drift_metadata)

Saved.
{'drift_point_row_index': 59054, 'drift_point_transaction_dt': 13990941, 'fraud_rate_multiplier': 2.5, 'amount_multiplier_for_fraud': 1.8, 'random_seed': 42, 'before_fraud_rate': 0.03134419345006265, 'after_fraud_rate': 0.09367697361736715, 'before_auc': 0.9070436192757474, 'after_auc': 0.6878883648388835}
